# Treadmill Spatiotemporal Gait Analysis

This notebook extracts spatiotemporal gait parameters from 3D motion-capture data collected during treadmill walking. It detects heel-strike and toe-off events from heel and toe marker vertical (Y) velocities, then computes per-cycle step time, step length, step width, stance/swing time and percentage, and stride time and length.

**Author**: Yeon-Joo Kang | Georgia State University | 2021–2024
**Status**: Archived. Reflects my analytical approach during PhD dissertation research.

---


## 1. Setup and Data Import

Load required packages, then read a single trial's motion-capture CSV. The CSV is expected to contain bilateral heel and toe marker trajectories in three dimensions: `LHEEX`, `LHEEY`, `LHEEZ`, `LTOEX`, `LTOEY`, `LTOEZ`, `RHEEX`, `RHEEY`, `RHEEZ`, `RTOEX`, `RTOEY`, `RTOEZ`. The Y axis corresponds to anterior-posterior direction (walking direction) and Z to vertical.

In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np
import os,sys
import seaborn as sns


Set working directory if the data files are not in the same folder as this notebook.

In [ ]:
# Check current working directory
os.getcwd()

In [ ]:
# Change directory if necessary
#newdirectory = input("Path: ")
#os.chdir(newdirectory)
#os.getcwd()

Read a trial by subject and trial number. The naming convention is `{Subject}Trial{N}.csv` (e.g., `S01Trial2.csv`).

In [ ]:
# File import by filename
Subject = input("Subject: ")
Trial = input("Trial: ")
filename = Subject + "Trial"+ Trial

df = pd.read_csv(filename +'.csv')
df.head()

Some exports start with a header row of zeros. If detected, drop it and reset the index.

In [ ]:
if any(df.iloc[0] == 0):
    df.drop([0], inplace=True)

df = df.reset_index(drop=True)
df.head()

## 2. Marker Velocity Calculation

Compute frame-to-frame vertical velocity for the four event-defining markers (`LHEEY`, `RHEEY`, `LTOEY`, `RTOEY`). The velocity is calculated as the position difference between consecutive frames, scaled by 100 (a constant kept consistent across this analysis; the absolute scale does not affect event detection since I rely on **sign change**, not magnitude).

In [ ]:
# Calculate Heel and Toe Marker Velocity
col_names = ['LHEEY','RHEEY', 'LTOEY', 'RTOEY']
name_rows_dict1 = {}

for col in col_names:
    vel_rows = []
    for idx, value in enumerate(df[col]):
        if idx == 0:
            vel_rows.append(0)
        else:
            vel_row = (df.loc[idx,col] - df.loc[idx-1,col])*100
            vel_rows.append(vel_row)
    name_rows_dict1[col] = vel_rows

vel_df = pd.DataFrame(name_rows_dict1)
vel_df = vel_df.drop(0)
vel_df = vel_df.reset_index(drop=True)
vel_df.head(10)

## 3. Left Foot Gait Event Detection

### Reasoning

I detect heel-strike (HS) and toe-off (TO) events using **velocity sign changes** rather than absolute position thresholds. The intuition:

- **Heel strike**: The heel marker is descending (negative velocity), reaches the ground, then begins rising. The frame where vertical heel velocity transitions from negative to non-negative is the HS event.
- **Toe off**: The toe marker is on the ground (small/positive velocity), then lifts off. The frame where vertical toe velocity transitions from positive to non-positive is the TO event.

This approach is robust across subjects without per-trial threshold tuning.

In [ ]:
# Identify Left Heel Strike
# by Left Heel marker velocity changes its direction
name_rows_dict2 = {}

lstrike_rows = []
for idx, value in enumerate(vel_df['LHEEY']):
    if idx == 0:
        idx = idx + 1
    else:
        if vel_df.loc[idx, 'LHEEY'] < 0:
            idx = idx + 1 
        else:
            if vel_df.loc[idx - 1, 'LHEEY'] > 0:
                idx = idx + 1 
            else:
                lstrike_rows.append(idx - 1)
            
name_rows_dict2['LHSidx'] = lstrike_rows

lstrike_df = pd.DataFrame(name_rows_dict2)
lstrike_df.head(), len(lstrike_df)

In [ ]:
# Identify Left Toe Off
# by Left Toe marker velocity changes its direction
name_rows_dict3 = {}

ltoff_rows = []
for idx, value in enumerate(vel_df['LTOEY']):
    if idx == 0:
        idx = idx + 1
    else:
        if vel_df.loc[idx, 'LTOEY'] > 0:
            idx = idx + 1 
        else:
            if vel_df.loc[idx - 1, 'LTOEY'] < 0:
                
                
                idx = idx + 1 
            else:
                ltoff_rows.append(idx - 1)
            
name_rows_dict3['LTOidx'] = ltoff_rows

ltoff_df = pd.DataFrame(name_rows_dict3)
ltoff_df.head(), len(ltoff_df)

### Filter false detections

Vertical-velocity sign changes can trigger spuriously on noisy frames. I filter out any pair of consecutive events that occur within fewer than 55 frames of each other. At 100 Hz sampling, 55 frames ≈ 0.55 s, well below the shortest physiologically plausible step-to-step interval. The first event in each list is assigned the default 55 to satisfy the interval check at the boundary.

In [ ]:
# Identify left side repeated Heel Strike by its interval (first interval default 55)

lefts = []
for idx in lstrike_df.index:
    if idx == 0:
        lefts.append(55)
    else:
        left = (lstrike_df.loc[idx, 'LHSidx']- lstrike_df.loc[idx - 1, 'LHSidx'])
        lefts.append(left)

lstrike_df['lHSinterval'] = lefts
lstrike_df.head(), lstrike_df['lHSinterval'].unique(), lstrike_df.count()

In [ ]:
# Identify left side repeated Toe Off by its interval (first interval default 55)

lefts = []
for idx in ltoff_df.index:
    if idx == 0:
        lefts.append(55)
    else:
        left = (ltoff_df.loc[idx, 'LTOidx']- ltoff_df.loc[idx - 1, 'LTOidx'])
        lefts.append(left)

ltoff_df['lTOinterval'] = lefts
ltoff_df.head(), ltoff_df['lTOinterval'].unique(), ltoff_df.count()

Drop events that fail the minimum-interval filter.

In [ ]:
#Drop left side repeated Heel Strike
for idx, row in lstrike_df.iterrows():
    if row['lHSinterval'] < 55:
        lstrike_df.drop(idx, inplace=True)


lstrike_df.head(), lstrike_df['lHSinterval'].unique(), lstrike_df.count()

In [ ]:
#Drop left side repeated Toe Off
for idx, row in ltoff_df.iterrows():
    if row['lTOinterval'] < 55:
        ltoff_df.drop(idx, inplace=True)


ltoff_df.head(), ltoff_df['lTOinterval'].unique(), ltoff_df.count()

Combine HS and TO event indices for the left foot into a single dataframe and align them.

In [ ]:
#Concate HS and TO event time & make sure them to be int 64
lstrike_df = pd.concat([lstrike_df,ltoff_df], axis=1)
lstrike_df = lstrike_df.astype('Int64')
lstrike_df.head()

In [ ]:
# Reset the index for left foot
lstrike_df = lstrike_df.reset_index(drop=True)
lstrike_df = lstrike_df.dropna()
lstrike_df.head(), len(lstrike_df)

Pull the X and Y trajectory values of the heel and toe markers at each event time, which are needed downstream for step length and step width.

In [ ]:
# Get trajectory for left heelstrike and left toeoff
LHSval = pd.DataFrame(df.loc[lstrike_df['LHSidx'], ['LHEEX','LHEEY']])
LHSval_reidx = LHSval.reset_index().drop(columns=['index'])

LTOval = pd.DataFrame(df.loc[lstrike_df['LTOidx'], ['LTOEY','LTOEZ']])
LTOval_reidx = LTOval.reset_index().drop(columns=['index'])


In [ ]:
# Concate the left HS and TO
LHS_df = pd.concat([lstrike_df, LHSval_reidx, LTOval_reidx], axis=1)
LHS_df.head()

## 4. Right Foot Gait Event Detection

Apply the same velocity-sign-change detection and interval filtering to the right foot.

In [ ]:
#NEW!!!! #Identify right heel strike
name_rows_dict4 = {}

rstrike_rows = []
for idx, value in enumerate(vel_df['RHEEY']):
    if idx == 0:
        idx = idx + 1
    else:
        if vel_df.loc[idx, 'RHEEY'] < 0:
            idx = idx + 1 
        else:
            if vel_df.loc[idx - 1, 'RHEEY'] > 0:
                idx = idx + 1 
            else:
                rstrike_rows.append(idx - 1)
            
name_rows_dict4['RHSidx'] = rstrike_rows

rstrike_df = pd.DataFrame(name_rows_dict4)
rstrike_df.head(), len(rstrike_df)

In [ ]:
# Identify Right Toe Off
# by Right Toe marker velocity changes its direction from positive to negative
name_rows_dict5 = {}

rtoff_rows = []
for idx, value in enumerate(vel_df['RTOEY']):
    if idx == 0:
        idx = idx + 1
    else:
        if vel_df.loc[idx, 'RTOEY'] > 0:
            idx = idx + 1 
        else:
            if vel_df.loc[idx - 1, 'RTOEY'] < 0:
                idx = idx + 1 
            else:
                rtoff_rows.append(idx - 1)
            
name_rows_dict5['RTOidx'] = rtoff_rows

rtoff_df = pd.DataFrame(name_rows_dict5)
rtoff_df.head(), len(rtoff_df)

Filter false detections on the right side using the same 55-frame minimum interval.

In [ ]:
# Identify right side repeated Heel Strike (first interval default 55)

rights = []
for idx in rstrike_df.index:
    if idx == 0:
        rights.append(55)
    else:
        right = (rstrike_df.loc[idx, 'RHSidx']- rstrike_df.loc[idx - 1, 'RHSidx'])
        rights.append(right)

rstrike_df['rHSinterval'] = rights
rstrike_df.head(), rstrike_df['rHSinterval'].unique(),  rstrike_df.count()

In [ ]:
# Identify right side repeated Toe Off (first interval default 55)

rights = []
for idx in rtoff_df.index:
    if idx == 0:
        rights.append(55)
    else:
        right = (rtoff_df.loc[idx, 'RTOidx']- rtoff_df.loc[idx - 1, 'RTOidx'])
        rights.append(right)

rtoff_df['rTOinterval'] = rights
rtoff_df.head(), rtoff_df['rTOinterval'].unique(),  rtoff_df.count()

In [ ]:
#Drop right side repeated Heel Strike
for idx, row in rstrike_df.iterrows():
    if row['rHSinterval'] < 55:
        rstrike_df.drop(idx, inplace=True)

rstrike_df.head(), rstrike_df['rHSinterval'].unique(), rstrike_df.count()

In [ ]:
#Drop right side repeated Toe Off
for idx, row in rtoff_df.iterrows():
    if row['rTOinterval'] < 55:
        rtoff_df.drop(idx, inplace=True)

rtoff_df.head(), rtoff_df['rTOinterval'].unique(), rtoff_df.count()

Combine right-foot HS and TO indices, then extract trajectory values at each event.

In [ ]:
#Concate HS and TO event time & make sure them to be int 64
rstrike_df = pd.concat([rstrike_df,rtoff_df], axis=1)
rstrike_df = rstrike_df.astype('Int64')
rstrike_df.head()

In [ ]:
# Reset the index for right foot
rstrike_df = rstrike_df.reset_index(drop=True)
rstrike_df = rstrike_df.dropna()
rstrike_df.head(), len(rstrike_df)

In [ ]:
# Get trajectory for left heelstrike and left toeoff
RHSval = pd.DataFrame(df.loc[rstrike_df['RHSidx'], ['RHEEX','RHEEY']])
RHSval_reidx = RHSval.reset_index().drop(columns=['index'])

RTOval = pd.DataFrame(df.loc[rstrike_df['RTOidx'], ['RTOEY','RTOEZ']])
RTOval_reidx = RTOval.reset_index().drop(columns=['index'])


In [ ]:
# Concate the right HS and TO
RHS_df = pd.concat([rstrike_df, RHSval_reidx, RTOval_reidx], axis=1)
RHS_df.head()

## 5. Merge Bilateral Events

Combine left and right foot events row-by-row by their event index, producing a single dataframe with both feet aligned on the same gait cycle.

In [ ]:
# Merge left and right side HS and TO 
strike_df = pd.merge(LHS_df, RHS_df, left_index = True, right_index = True)
strike_df.head()

## 6. Spatiotemporal Parameters

### Step time

Step time is the duration between the contralateral and ipsilateral heel-strike events. The branching logic handles whichever foot strikes first in the trial (determined by the very first row), since `Rstep_time` and `Lstep_time` must alternate accordingly throughout the trial.

In [ ]:
# Calculate step time


Rsteptime = []
Lsteptime = []

for idx in strike_df.index:
    if strike_df.loc[0, 'LHSidx'] < strike_df.loc[0, 'RHSidx']:
        if idx == 0:
            Rstep_time = strike_df.loc[idx, 'RHSidx'] - strike_df.loc[idx, 'LHSidx']
            Rsteptime.append(Rstep_time)
            Lsteptime.append(0)
        else:
            Rstep_time = strike_df.loc[idx, 'RHSidx'] - strike_df.loc[idx, 'LHSidx']
            Rsteptime.append(Rstep_time)
            Lstep_time = strike_df.loc[idx, 'LHSidx'] - strike_df.loc[idx - 1,'RHSidx']
            Lsteptime.append(Lstep_time)
    else:
        if idx ==0:
            Rsteptime.append(0)
            Lstep_time = strike_df.loc[idx, 'LHSidx'] - strike_df.loc[idx,'RHSidx']
            Lsteptime.append(Lstep_time)
        else:
            Rstep_time = strike_df.loc[idx, 'RHSidx'] - strike_df.loc[idx - 1, 'LHSidx']
            Rsteptime.append(Rstep_time)
            Lstep_time = strike_df.loc[idx, 'LHSidx'] - strike_df.loc[idx,'RHSidx']
            Lsteptime.append(Lstep_time)
        

strike_df['Rstep_time'] = Rsteptime
strike_df['Lstep_time'] = Lsteptime

strike_df.head()

### Treadmill belt speed

Step length on a treadmill cannot be measured purely from marker positions because the body's anterior position is held roughly constant by the belt. The belt's forward displacement during each step must be added back. Belt speed is stored per subject in `input_val.csv`.

In [ ]:
# Get Treadmil Belt Speed
input_val = pd.read_csv('input_val.csv')
input_val = input_val.set_index('Unnamed: 0')
beltspeed = input_val.loc[Subject, 'belt_speed']
beltspeed, type(beltspeed)

### Step length

Step length combines the anterior-posterior heel position difference with the belt-displacement correction:

```
step_length = (contralateral heel Y at HS) - (ipsilateral heel Y at previous HS) + (belt speed × step time)
```

The factor of 10 in the code converts the time/speed units to match marker position units.

In [ ]:
#Calculate Step Length

Rsteplength = []
Lsteplength = []

for idx in strike_df.index:
    if strike_df.loc[0, 'LHSidx'] < strike_df.loc[0, 'RHSidx']:
        if idx == 0:
            Rstep_length = strike_df.loc[idx, 'RHEEY'] - (strike_df.loc[idx, 'LHEEY'] - (beltspeed*strike_df.loc[idx, 'Rstep_time']*10))
            Rsteplength.append(Rstep_length)
            Lsteplength.append(0)
        else:
            Rstep_length = strike_df.loc[idx, 'RHEEY'] - (strike_df.loc[idx, 'LHEEY'] - (beltspeed*strike_df.loc[idx, 'Rstep_time']*10))
            Rsteplength.append(Rstep_length)
            Lstep_length = strike_df.loc[idx, 'LHEEY'] - (strike_df.loc[idx - 1,'RHEEY']-(beltspeed*strike_df.loc[idx, 'Lstep_time']*10))
            Lsteplength.append(Lstep_length)
    else:
        if idx ==0:
            Rsteplength.append(0)
            Lstep_length = strike_df.loc[idx, 'LHEEY'] - (strike_df.loc[idx,'RHEEY']-(beltspeed*strike_df.loc[idx, 'Lstep_time']*10))
            Lsteplength.append(Lstep_length)
        else:
            Rstep_length = strike_df.loc[idx, 'RHEEY'] - (strike_df.loc[idx - 1, 'LHEEY'] - (beltspeed*strike_df.loc[idx, 'Rstep_time']*10))
            Rsteplength.append(Rstep_length)
            Lstep_length = strike_df.loc[idx, 'LHEEY'] - (strike_df.loc[idx,'RHEEY']-(beltspeed*strike_df.loc[idx, 'Lstep_time']*10))
            Lsteplength.append(Lstep_length)
        

strike_df['Rstep_length'] = Rsteplength
strike_df['Lstep_length'] = Lsteplength

strike_df.tail()

### Step width

Lateral (X-axis) distance between contralateral heel markers at HS events.

In [ ]:
#Calculate Step Width

Rstepwidth = []
Lstepwidth = []

for idx in strike_df.index:
    if strike_df.loc[0, 'LHSidx'] < strike_df.loc[0, 'RHSidx']:
        if idx == 0:
            Rstep_width = abs(strike_df.loc[idx, 'RHEEX'] - (strike_df.loc[idx, 'LHEEX']))
            Rstepwidth.append(Rstep_width)
            Lstepwidth.append(0)
        else:
            Rstep_width = abs(strike_df.loc[idx, 'RHEEX'] - (strike_df.loc[idx, 'LHEEX']))
            Rstepwidth.append(Rstep_width)
            Lstep_width = abs(strike_df.loc[idx, 'LHEEX'] - (strike_df.loc[idx - 1,'RHEEX']))
            Lstepwidth.append(Lstep_width)
    else:
        if idx ==0:
            Rstepwidth.append(0)
            Lstep_width = abs(strike_df.loc[idx, 'LHEEX'] - (strike_df.loc[idx,'RHEEX']))
            Lstepwidth.append(Lstep_width)
        else:
            Rstep_width = abs(strike_df.loc[idx, 'RHEEX'] - (strike_df.loc[idx - 1, 'LHEEX']))
            Rstepwidth.append(Rstep_width)
            Lstep_width = abs(strike_df.loc[idx, 'LHEEX'] - (strike_df.loc[idx,'RHEEX']))
            Lstepwidth.append(Lstep_width)
        

strike_df['Rstep_width'] = Rstepwidth
strike_df['Lstep_width'] = Lstepwidth

strike_df.tail()

### Stance and swing time

Stance time spans from one foot's HS to its own TO. Swing time spans from TO back to the next HS on the same side.

In [ ]:
# Calculate Stance time


Rstancetime = []
Lstancetime = []

for idx in strike_df.index:
    if strike_df.loc[0, 'LHSidx'] < strike_df.loc[0, 'LTOidx']:
        Lstance_time = strike_df.loc[idx, 'LTOidx'] - strike_df.loc[idx, 'LHSidx']
        Lstancetime.append(Lstance_time)
        
    else:
        if idx == 0:
            Lstancetime.append(0)
        else:
            Lstance_time = strike_df.loc[idx, 'LTOidx'] - strike_df.loc[idx - 1, 'LHSidx']
            Lstancetime.append(Lstance_time)

for idx in strike_df.index:
    if strike_df.loc[0, 'RHSidx'] < strike_df.loc[0, 'RTOidx']:
        Rstance_time = strike_df.loc[idx, 'RTOidx'] - strike_df.loc[idx, 'RHSidx']
        Rstancetime.append(Rstance_time)
        
    else:
        if idx == 0:
            Rstancetime.append(0)
        else:
            Rstance_time = strike_df.loc[idx, 'RTOidx'] - strike_df.loc[idx - 1, 'RHSidx']
            Rstancetime.append(Rstance_time)
            

strike_df['Rstance_time'] = Rstancetime
strike_df['Lstance_time'] = Lstancetime

strike_df.head()

In [ ]:
# Calculate Swing time


Rswingtime = []
Lswingtime = []

for idx in strike_df.index:
    if strike_df.loc[0, 'LHSidx'] < strike_df.loc[0, 'LTOidx']:
        if idx == 0:
            Lswingtime.append(0)
        else:
            Lswing_time = strike_df.loc[idx, 'LHSidx'] - strike_df.loc[idx - 1,'LTOidx']
            Lswingtime.append(Lswing_time)
    else:
        Lswing_time = strike_df.loc[idx, 'LHSidx'] - strike_df.loc[idx,'LTOidx']
        Lswingtime.append(Lswing_time)

for idx in strike_df.index:
    if strike_df.loc[0, 'RHSidx'] < strike_df.loc[0, 'RTOidx']:
        if idx == 0:
            Rswingtime.append(0)
        else:
            Rswing_time = strike_df.loc[idx, 'RHSidx'] - strike_df.loc[idx - 1,'RTOidx']
            Rswingtime.append(Rswing_time)
    else:
        Rswing_time = strike_df.loc[idx, 'RHSidx'] - strike_df.loc[idx,'RTOidx']
        Rswingtime.append(Rswing_time)
        
        

strike_df['Rswing_time'] = Rswingtime
strike_df['Lswing_time'] = Lswingtime

strike_df.head()

### Stance and swing percentage of gait cycle

Express stance and swing as percentages of the gait cycle, using stance time of the previous cycle and swing time of the current cycle (or vice versa depending on which event came first).

In [ ]:
# Caculate stance and swing in %

Rstance = []
Lstance = []
Rswing = []
Lswing = []


for idx in strike_df.index:
    if strike_df.loc[0, 'LHSidx'] < strike_df.loc[0, 'LTOidx']:
        if idx == 0:
            Lstance.append(0)
            Lswing.append(0) 
        else:
            Lstance_per = (strike_df.loc[idx - 1, 'Lstance_time']/(strike_df.loc[idx, 'Lswing_time'] + strike_df.loc[idx - 1,'Lstance_time']))*100
            Lstance.append(Lstance_per)
            Lswing_per = (strike_df.loc[idx, 'Lswing_time']/(strike_df.loc[idx, 'Lswing_time'] + strike_df.loc[idx - 1,'Lstance_time']))*100
            Lswing.append(Lswing_per)
    else:
        if idx == 0:
            Lstance.append(0)
            Lswing.append(0) 
        else:
            Lstance_per = (strike_df.loc[idx, 'Lstance_time']/(strike_df.loc[idx - 1, 'Lswing_time'] + strike_df.loc[idx,'Lstance_time']))*100
            Lstance.append(Lstance_per)
            Lswing_per = (strike_df.loc[idx - 1, 'Lswing_time']/(strike_df.loc[idx - 1, 'Lswing_time'] + strike_df.loc[idx,'Lstance_time']))*100
            Lswing.append(Lswing_per)

for idx in strike_df.index:
    if strike_df.loc[0, 'RHSidx'] < strike_df.loc[0, 'RTOidx']:
        if idx == 0:
            Rstance.append(0)
            Rswing.append(0) 
        else:
            Rstance_per = (strike_df.loc[idx - 1, 'Rstance_time']/(strike_df.loc[idx, 'Rswing_time'] + strike_df.loc[idx - 1,'Rstance_time']))*100
            Rstance.append(Rstance_per)
            Rswing_per = (strike_df.loc[idx, 'Rswing_time']/(strike_df.loc[idx, 'Rswing_time'] + strike_df.loc[idx - 1,'Rstance_time']))*100
            Rswing.append(Rswing_per)
    else:
        if idx == 0:
            Rstance.append(0)
            Rswing.append(0) 
        else:
            Rstance_per = (strike_df.loc[idx, 'Rstance_time']/(strike_df.loc[idx - 1, 'Rswing_time'] + strike_df.loc[idx,'Rstance_time']))*100
            Rstance.append(Rstance_per)
            Rswing_per = (strike_df.loc[idx - 1, 'Rswing_time']/(strike_df.loc[idx - 1, 'Rswing_time'] + strike_df.loc[idx,'Rstance_time']))*100
            Rswing.append(Rswing_per)
        

strike_df['Rstance%'] = Rstance
strike_df['Lstance%'] = Lstance
strike_df['Rswing%'] = Rswing
strike_df['Lswing%'] = Lswing

strike_df.head()

### Stride time and stride length

Stride time spans two consecutive ipsilateral HS events. Stride length applies the same belt-displacement correction as step length.

In [ ]:
# Calculate stride time

Rstridetime = []
Lstridetime = []

for idx in strike_df.index:
    if strike_df.loc[0, 'LHSidx'] < strike_df.loc[0, 'RHSidx']:
        if idx == 0:
            Rstridetime.append(0)
            Lstridetime.append(0)
        else:
            Rstride_time = strike_df.loc[idx, 'Lstep_time'] + strike_df.loc[idx, 'Rstep_time']
            Rstridetime.append(Rstride_time)
            Lstride_time = strike_df.loc[idx - 1, 'Rstep_time'] + strike_df.loc[idx, 'Lstep_time']
            Lstridetime.append(Lstride_time)
    else:
        if idx ==0:
            Rstridetime.append(0)
            Lstridetime.append(0)
        else:
            Rstride_time = strike_df.loc[idx - 1, 'Lstep_time'] + strike_df.loc[idx, 'Rstep_time']
            Rstridetime.append(Rstride_time)
            Lstride_time = strike_df.loc[idx, 'Rstep_time'] + strike_df.loc[idx, 'Lstep_time']
            Lstridetime.append(Lstride_time)
        

strike_df['Rstride_time'] = Rstridetime
strike_df['Lstride_time'] = Lstridetime

strike_df.head()

In [ ]:
# Calculate stride length

Rstridelength = []
Lstridelength = []

for idx in strike_df.index:
    if strike_df.loc[0, 'LHSidx'] < strike_df.loc[0, 'RHSidx']:
        if idx == 0:
            Rstridelength.append(0)
            Lstridelength.append(0)
        else:
            Rstride_length = strike_df.loc[idx, 'Lstep_length'] + strike_df.loc[idx, 'Rstep_length']
            Rstridelength.append(Rstride_length)
            Lstride_length = strike_df.loc[idx - 1, 'Rstep_length'] + strike_df.loc[idx, 'Lstep_length']
            Lstridelength.append(Lstride_length)
    else:
        if idx ==0:
            Rstridelength.append(0)
            Lstridelength.append(0)
        else:
            Rstride_length = strike_df.loc[idx - 1, 'Lstep_length'] + strike_df.loc[idx, 'Rstep_length']
            Rstridelength.append(Rstride_length)
            Lstride_length = strike_df.loc[idx, 'Rstep_length'] + strike_df.loc[idx, 'Lstep_length']
            Lstridelength.append(Lstride_length)
        

strike_df['Rstride_length'] = Rstridelength
strike_df['Lstride_length'] = Lstridelength

strike_df.head()

## 7. Output

Save per-cycle spatiotemporal values for this trial, then compute trial-level mean and SD for each parameter and append to a cross-trial aggregation CSV.

In [ ]:
# Save spatiotemporal data in csv 

strike_df.to_csv(filename + '_data.csv',  index=False)

Drop the first row if its values are zero (the boundary case from the alternation logic above), then compute trial-level summary.

In [ ]:
#Drop first row if it contains 0

if any(strike_df.iloc[0] == 0):
    strike_df.drop([0], inplace=True)

strike_df = strike_df.reset_index(drop=True)
strike_df.head()

In [ ]:
# Get Mean and SD for the  trial

step = ['Rstep_time','Lstep_time','Rstep_length','Lstep_length', 'Rstep_width', 'Lstep_width',
        'Rstance%', 'Lstance%', 'Rswing%', 'Lswing%',
        'Rstride_time', 'Lstride_time', 'Rstride_length', 'Lstride_length']
rows = []

for col in step:
    df_attr = getattr(strike_df, col)
    row = pd.DataFrame({col+"_Avg":[df_attr.mean()], 
                        col+"_SD":[df_attr.std()]
                       }, index = [filename])
    rows.append(row)

step_df = pd.concat(rows, axis=1)
step_df

Append the trial-level summary to the aggregate CSV, creating it on the first trial.

In [ ]:
if not os.path.exists('step1.csv'):
    step_df.to_csv('step1.csv', index=filename, mode='w')
else:
    step_df.to_csv('step1.csv', index=filename, mode='a')